#Trabajo de Sandra Blázquez Arriero

## Insertar los datos:

Lo primero que tenemos que hacer es insertar los datos con los que vamos a trabajar, en mi caso voy a insertarlos desde mi portatil, es decir, de manera local, porque es donde tengo los datos.

Para hacerlo al ejecutar este comando, nos saldrá un botón de elegir archivos, y con ese elegimos los archivos que tenemos en el portatil.

In [1]:
from google.colab import files
uploaded = files.upload()

Saving 1-s2.0-S1746809421006297-mmc1.zip to 1-s2.0-S1746809421006297-mmc1.zip


In [2]:
import os
os.listdir()

['.config', '1-s2.0-S1746809421006297-mmc1.zip', 'sample_data']

Ahora, como tenemos un archivo .zip, tenemos que extraer los archivos del zip.

In [3]:
import zipfile
import os

zip_file = '1-s2.0-S1746809421006297-mmc1.zip'

extract_path = 'extracted_files'
os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_file, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Archivos extraídos:")
print(os.listdir(extract_path))


Archivos extraídos:
['Data']


Vamos a ver todo lo que tenemos dentro de la carpeta para ver también donde se encuentran todos los archivos que vamos a tener que utilizar.

In [4]:
import os

data_directory = 'extracted_files/Data'

def listar_archivos(directorio):
    archivos = []
    for root, dirs, files in os.walk(directorio):
        for file in files:
            archivos.append(os.path.join(root, file))
    return archivos

# Mostrar los archivos
archivos_en_data = listar_archivos(data_directory)
for archivo in archivos_en_data:
    print(archivo)


extracted_files/Data/channels.csv
extracted_files/Data/readme.txt
extracted_files/Data/NBack/NBack_15.csv
extracted_files/Data/NBack/EEG_16.csv
extracted_files/Data/NBack/NBack_18.csv
extracted_files/Data/NBack/EEG_1.csv
extracted_files/Data/NBack/NBack_6.csv
extracted_files/Data/NBack/EEG_11.csv
extracted_files/Data/NBack/NBack_17.csv
extracted_files/Data/NBack/NBack_3.csv
extracted_files/Data/NBack/NBack_2.csv
extracted_files/Data/NBack/NBack_9.csv
extracted_files/Data/NBack/NBack_1.csv
extracted_files/Data/NBack/EEG_4.csv
extracted_files/Data/NBack/EEG_12.csv
extracted_files/Data/NBack/NBack_16.csv
extracted_files/Data/NBack/NBack_4.csv
extracted_files/Data/NBack/NBack_12.csv
extracted_files/Data/NBack/EEG_15.csv
extracted_files/Data/NBack/NBack_14.csv
extracted_files/Data/NBack/EEG_8.csv
extracted_files/Data/NBack/NBack_7.csv
extracted_files/Data/NBack/EEG_5.csv
extracted_files/Data/NBack/NBack_13.csv
extracted_files/Data/NBack/EEG_9.csv
extracted_files/Data/NBack/EEG_18.csv
extrac

## Cambio nombre en columnas:

Ahora, como en el archivo de EEG tenemos nombres no significativos, tenemos que reemplazar los nombres de las columnas de esos que teniamos a los que tenemos en el archivo "channels.csv" que son más significativos, los reescribimos y los guardamos en una nueva carpeta dentro de la anterior.

In [5]:
import pandas as pd
import os

channels_path = 'extracted_files/Data/channels.csv'
input_directory = 'extracted_files/Data/NBack'
output_directory = 'renamed_EEG_files'
os.makedirs(output_directory, exist_ok=True)

channels_df = pd.read_csv(channels_path)
channel_names = channels_df.columns.tolist()

eeg_files = [f for f in os.listdir(input_directory) if f.startswith('EEG') and f.endswith('.csv')]

for eeg_file in eeg_files:
    eeg_path = os.path.join(input_directory, eeg_file)
    eeg_df = pd.read_csv(eeg_path)

    for i in range(16):
        old_col = f'EEG{i}'
        if old_col in eeg_df.columns:
            eeg_df.rename(columns={old_col: channel_names[i]}, inplace=True)

    eeg_df.to_csv(os.path.join(output_directory, eeg_file), index=False)

print("Todos los archivos EEG renombrados y guardados en 'renamed_EEG_files'")


✅ Todos los archivos EEG renombrados y guardados en 'renamed_EEG_files'


## Combinar los datos:

Ahora, como tenemos 19 archivos distintos tanto de NBack como de EEG, tenemos que unir todos pero de una manera particular.

Lo que tenemos que hacer es, unir los archivos que coincidan con el numero (ejemplo NBack_1 con EEG_1 y así con todos).
Después de eso tenemos que ver el valor de la columna Time del archivo NBack correspondiente, ese time será con el cual elegiremos con cual de los EEG empezamos, por ejemplo, si es 17.92353 el time de NBack, entonces los valores de EEG con los que tenemos que trabajar seran los que tengan un Time igual o mayor a ese que tiene el NBack.

Cuando tengamos eso, separamos por bloques, y esto lo hacemos por el Event Code del NBack, el event code nos indica que, cuando sea 1, empieza el bloque, y ese acaba cuando el event code sea 2, y así con todos los bloques que se tengan por cada archivo (que en total creo que son 3 por cada uno)

Una vez tengamos eso con quedamos con los valores correspondientes unicamente a esos dos, el inicio y el fin del bloque, el resto los ignoramos.

Y eso es lo que hemos realizado en el código que vemos despues, tenemos las rutas de los archivos, y los canales que queremos del eeg, cogemos con el for los archicos que tenemos, que son del 1 al 19 y vamos cogiendo los valores necesarias para el data frame, esto, para ser mas sencillo lo dividimos en el inicio (event code 1) y fin (event code 2) y al finalizar los metemos todos en un data frame.

In [13]:
import pandas as pd
import os

eeg_dir = 'renamed_EEG_files'
nback_dir = 'extracted_files/Data/NBack'

filas_df = []

canales_eeg = ['F4','Oz','Fz','F8','Pz','F5','T5','T3','C4','C3','T4','P4','T6','Cz','P3','F7']

for s in range(1, 20):
    eeg_path = os.path.join(eeg_dir, f'EEG_{s}.csv')
    events_path = os.path.join(nback_dir, f'NBack_{s}.csv')

    try:
        eeg_df = pd.read_csv(eeg_path)
        events_df = pd.read_csv(events_path)

        eeg_df['Time'] = eeg_df['Time'].astype(float)
        events_df['Time'] = events_df['Time'].astype(float)

        # Buscar eventos de inicio y fin
        event_starts = events_df[events_df['Event Code'] == 1].index.tolist()
        event_ends = events_df[events_df['Event Code'] == 2].index.tolist()

        if len(event_starts) != len(event_ends):
            print(f"⚠️ Participante {s}: Eventos desbalanceados. Saltando.")
            continue

        for i, (start_idx, end_idx) in enumerate(zip(event_starts, event_ends)):
            start_time = events_df.loc[start_idx, 'Time']
            end_time = events_df.loc[end_idx, 'Time']

            eeg_start_idx = eeg_df[eeg_df['Time'] >= start_time].index.min()
            eeg_end_idx = eeg_df[eeg_df['Time'] <= end_time].index.max()

            # Fila de inicio
            fila_inicio = {
                'participante': s,
                'bloque': i + 1,
                'time eeg': eeg_df.loc[eeg_start_idx, 'Time'],
                'Notes': events_df.loc[start_idx, 'Notes'],
            }
            for canal in canales_eeg:
                fila_inicio[canal] = eeg_df.loc[eeg_start_idx, canal]

            # Fila de fin
            fila_fin = {
                'participante': s,
                'bloque': i + 1,
                'time eeg': eeg_df.loc[eeg_end_idx, 'Time'],
                'Notes': events_df.loc[end_idx, 'Notes'],
            }
            for canal in canales_eeg:
                fila_fin[canal] = eeg_df.loc[eeg_end_idx, canal]

            # Agregar ambas filas
            filas_df.append(fila_inicio)
            filas_df.append(fila_fin)

    except Exception as e:
        print(f"❌ Error en participante {s}: {e}")

# Convertir a DataFrame
df_eeg_notas = pd.DataFrame(filas_df)

print(df_eeg_notas.head())

   participante  bloque    time eeg  Notes           F4            Oz  \
0             1       1   17.924878    1.0  2314.685302  14760.401830   
1             1       1   95.025437    1.0  2576.716653  15532.527367   
2             1       2  108.525945    2.0  2500.425194  15013.976631   
3             1       2  185.556188    2.0  2913.494166  15412.432199   
4             1       3  229.417213    3.0  2606.150312  14439.880360   

             Fz            F8           Pz            F5            T5  \
0  19244.003426  15623.870267  3445.551045  17231.466371  23761.431219   
1  19172.019331  15483.868747  4255.092250  17107.051671  24047.870970   
2  19179.977005  15343.636041  3925.724864  17055.351127  23662.313161   
3  19354.705131  15609.816577  4206.105102  17189.195792  23774.122127   
4  18870.601146  15113.155528  3682.504773  16834.629107  23393.577409   

            T3            C4            C3            T4            P4  \
0  8791.939687  19545.786642  25822.797241

Este es el dataframe terminado:

In [12]:
df_eeg_notas

,participante,bloque,time eeg,Notes,F4,Oz,Fz,F8,Pz,F5,T5,T3,C4,C3,T4,P4,T6,Cz,P3,F7
0,1,1,17.924878,1.0,2314.685302,14760.401830,19244.003426,15623.870267,3445.551045,17231.466371,23761.431219,8791.939687,19545.786642,25822.797241,19724.517941,18545.504622,41608.765565,4528.658577,14541.736150,18810.383214
1,1,1,95.025437,1.0,2576.716653,15532.527367,19172.019331,15483.868747,4255.092250,17107.051671,24047.870970,8219.607732,19533.910970,25682.430690,19969.356336,19004.591823,42067.731089,4586.856671,15531.383603,18588.322747
2,1,2,108.525945,2.0,2500.425194,15013.976631,19179.977005,15343.636041,3925.724864,17055.351127,23662.313161,8269.945493,19423.464786,25542.940214,19753.781252,18648.930045,41730.880570,4495.197411,15046.464382,18621.978596
3,1,2,185.556188,2.0,2913.494166,15412.432199,19354.705131,15609.816577,4206.105102,17189.195792,23774.122127,8137.463610,19589.566016,25420.022141,19922.729722,19021.480586,42146.979299,4744.915053,15425.585480,18928.641060
4,1,3,229.417213,3.0,2606.150312,14439.880360,18870.601146,15113.155528,3682.504773,16834.629107,23393.577409,7788.627910,19162.857056,24896.032445,19657.498267,18347.329344,41421.370866,4279.281631,14930.275045,18741.234193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,19,1,91.768567,1.0,-8689.341668,5422.096052,2162.649930,-13275.529116,2848.591671,4806.276748,-4056.381593,-14537.854655,4855.178722,898.998112,-21384.921849,3849.299561,-2518.019684,7268.154679,-5159.249465,-3339.533902
110,19,2,103.712689,2.0,-8769.624131,5462.188613,2130.198683,-13309.562164,2749.278929,4823.834735,-4040.673096,-14429.002439,4768.897584,931.181670,-21473.405341,3773.543480,-2544.727778,7183.698695,-5210.049600,-3215.423395
111,19,2,180.780343,2.0,-8483.512908,6100.128790,2382.909578,-12113.952106,2948.013922,5048.754610,-3594.106455,-13775.511946,4856.018293,1332.082945,-20838.434082,3898.773416,-2271.806338,7306.495091,-4972.487487,-2629.487954
112,19,3,198.185860,3.0,-8543.414480,6288.922774,2296.056558,-12068.079889,2944.424451,4943.686548,-3512.814073,-13681.394811,4836.075438,1364.485522,-20690.596566,3901.596322,-2158.366901,7241.361410,-4960.599647,-2741.272585
